In [4]:

import sys, os

PROJECT_ROOT = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen"
SEG_RDM_ROOT = f"{PROJECT_ROOT}/scripts/SEG-RDM"

if SEG_RDM_ROOT not in sys.path:
    sys.path.insert(0, SEG_RDM_ROOT)

print("sys.path[0]:", sys.path[0])
print("Python:", sys.version.split()[0])


sys.path[0]: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM
Python: 3.9.23


In [5]:

# ── Real training settings (mirrors submit_rdm_train.sh exactly) ─────────────
PROJECT_ROOT = "/scratch/gilbreth/abelde/Thesis/StructureAwareGen"
RDM_ROOT     = f"{PROJECT_ROOT}/scripts/SEG-RDM/rdm"
RUN_DIR      = f"{RDM_ROOT}/rdm_out_final/IJEPA_local_feat/debug"

os.makedirs(RUN_DIR, exist_ok=True)

args_dict = dict(
    # ── paths ────────────────────────────────────────────────────────────────
    data_path         = f"{PROJECT_ROOT}/dataset/imagenet-1K-hf",
    h5_path           = f"{PROJECT_ROOT}/h5_embeddings/region_emb_flat.h5",
    ijepa_h5_path     = f"{PROJECT_ROOT}/h5_embeddings/ijepa_emb_flat.h5",
    ijepa_lookup_json = f"{PROJECT_ROOT}/h5_embeddings/ijepa_lookup.json",
    mask_npz_dir      = None,          # flat H5 mode — no NPZ dir needed
    max_segments      = 250,
    emb_source        = "region",

    # ── model ─────────────────────────────────────────────────────────────────
    config            = f"{RDM_ROOT}/configs/unified_seg_rdm_region_ijepa.yaml",
    input_size        = 256,

    # ── training (debug-friendly values) ─────────────────────────────────────
    batch_size        = 2,             # small for debug
    accum_iter        = 1,
    blr               = 5e-7,
    min_lr            = 1e-6,
    cosine_lr         = True,
    warmup_epochs     = 5,
    weight_decay      = 0.01,
    epochs            = 200,
    num_workers       = 0,             # 0 = no multiprocessing in notebook
    output_dir        = RUN_DIR,
    log_dir           = RUN_DIR,

    # ── distributed (single GPU in notebook) ─────────────────────────────────
    world_size        = 1,
    rank              = 0,
    local_rank        = 0,
    dist_url          = "env://",
    device            = "cuda" if __import__("torch").cuda.is_available() else "cpu",
    seed              = 42,
    resume            = "",
)

print("Settings ready.")
print(f"  Region H5     : {args_dict['h5_path']}")
print(f"  IJEPA H5      : {args_dict['ijepa_h5_path']}")
print(f"  IJEPA lookup  : {args_dict['ijepa_lookup_json']}")
print(f"  Data path     : {args_dict['data_path']}")
print(f"  Device        : {args_dict['device']}")
print(f"  Batch size    : {args_dict['batch_size']}")


Settings ready.
  Region H5     : /scratch/gilbreth/abelde/Thesis/StructureAwareGen/h5_embeddings/region_emb_flat.h5
  IJEPA H5      : /scratch/gilbreth/abelde/Thesis/StructureAwareGen/h5_embeddings/ijepa_emb_flat.h5
  IJEPA lookup  : /scratch/gilbreth/abelde/Thesis/StructureAwareGen/h5_embeddings/ijepa_lookup.json
  Data path     : /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet-1K-hf
  Device        : cuda
  Batch size    : 2


In [6]:

# ── Build dataset ─────────────────────────────────────────────────────────────
from rdm.data.seg_dataset import SegmentationMaskDataset

ds = SegmentationMaskDataset(
    image_dir         = os.path.join(args_dict["data_path"], "train"),
    mask_npz_dir      = args_dict["mask_npz_dir"],
    max_segments      = args_dict["max_segments"],
    image_size        = args_dict["input_size"],
    emb_source        = args_dict["emb_source"],
    h5_path           = args_dict["h5_path"],
    ijepa_h5_path     = args_dict["ijepa_h5_path"],
    ijepa_lookup_json = args_dict["ijepa_lookup_json"],
    normalize         = True,
)

print(f"\nDataset size : {len(ds):,} samples")

# ── Inspect one sample ────────────────────────────────────────────────────────
sample = ds[0]
print("\nSample keys :", list(sample.keys()))
for k, v in sample.items():
    if hasattr(v, "shape"):
        print(f"  {k:20s} shape={v.shape}  dtype={v.dtype}")
    else:
        print(f"  {k:20s} {v}")


Using flat .h5 file: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/h5_embeddings/region_emb_flat.h5
Using flat IJEPA h5 with lookup: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/h5_embeddings/ijepa_lookup.json (1281167 entries, loaded in 0.9s)
Building image lookup table from /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet-1K-hf/train...
  Found 1281167 images
[Flat H5 discovery] Reading index from /scratch/gilbreth/abelde/Thesis/StructureAwareGen/h5_embeddings/region_emb_flat.h5 ...
  Indexed 510172 samples in 63.2s
  Matched 510172/1281167 images in 66.6s (39.8% coverage)
SegmentationMaskDataset: Loaded 510172 images with region embeddings
  image_dir: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/dataset/imagenet-1K-hf/train
  mask_npz_dir: None
  h5_path: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/h5_embeddings/region_emb_flat.h5
  max_segments: 250
  emb_source: region

Dataset size : 510,172 samples

Sample keys : ['image', 'seg_embs', 'num_

In [8]:

# ── Build DataLoader and pull one batch ───────────────────────────────────────
import torch
from torch.utils.data import DataLoader
from rdm.data.seg_dataset import collate_seg_batch

loader = DataLoader(
    ds,
    batch_size  = args_dict["batch_size"],
    shuffle     = True,
    num_workers = args_dict["num_workers"],
    collate_fn  = collate_seg_batch,
    pin_memory  = False,
)

batch = next(iter(loader))

print("Batch keys:", list(batch.keys()))
for k, v in batch.items():
    if isinstance(v, torch.Tensor):
        print(f"  {k:20s} shape={list(v.shape)}  dtype={v.dtype}")
    else:
        print(f"  {k:20s} {v}")


Batch keys: ['image', 'seg_embs', 'num_segments', 'scores', 'filename', 'emb_source', 'ijepa_emb']
  image                shape=[2, 3, 256, 256]  dtype=torch.float32
  seg_embs             shape=[2, 250, 256]  dtype=torch.float32
  num_segments         shape=[2]  dtype=torch.int64
  scores               shape=[2, 42]  dtype=torch.float32
  filename             ['337159', '122474']
  emb_source           region
  ijepa_emb            shape=[2, 1280]  dtype=torch.float32


In [9]:

# ── Build model from the real config ─────────────────────────────────────────
from omegaconf import OmegaConf
from rdm import util

config = OmegaConf.load(args_dict["config"])
print("Config loaded:", args_dict["config"])
print("Model target :", config.model.target)

device = torch.device(args_dict["device"])
model  = util.instantiate_from_config(config.model)
model  = model.to(device)
model.train()

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nModel parameters : {n_params:,}")
print(f"Running on       : {device}")


[env] module=rdm.util
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] numpy=1.20.3
[env] torch=2.7.0+cu126
[env] torch.cuda=12.6
[env] cudnn=90501
Config loaded: /scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/configs/unified_seg_rdm_region_ijepa.yaml
Model target : rdm.models.diffusion.ddpm.UnifiedSegRDM
[env] module=rdm.models.diffusion
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] module=rdm.pretrained_enc.moco_v3
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] module=rdm.pretrained_enc.moco_v3.vits
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] torch=2.7.0+cu126
[env] torch.cuda=12.6
[env] cudnn=90501


/home/abelde/.local/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[env] module=rdm.pretrained_enc.dino
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] module=rdm.pretrained_enc.dino.vits
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] torch=2.7.0+cu126
[env] torch.cuda=12.6
[env] cudnn=90501
[env] module=rdm.pretrained_enc.ibot
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] module=rdm.pretrained_enc.ibot.vits
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] torch=2.7.0+cu126
[env] torch.cuda=12.6
[env] cudnn=90501
[env] module=rdm.pretrained_enc.deit
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] module=rdm.pretrained_enc.deit.vits
[env] python=3.9.23 | packaged by conda-forge | (main, Jun  4 2025, 17:57:12)  [GCC 13.3.0]
[env] torch=2.7.0+cu126
[env] torch.cuda=12.6
[env] cudnn=90501
[env] module=rdm.pr

/home/abelde/.local/lib/python3.9/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)
/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/pretrained_enc/deit/vits.py:70: UserWarning: Overwriting deit_tiny_patch16_224 in registry with rdm.pretrained_enc.deit.vits.deit_tiny_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  def deit_tiny_patch16_224(pretrained=False, **kwargs):
/scratch/gilbreth/abelde/Thesis/StructureAwareGen/scripts/SEG-RDM/rdm/pretrained_enc/deit/vits.py:85: UserWarning: Overwriting deit_small_patch16_224 in registry with rdm.pretrained_enc.deit.vits.deit_small_patch16_224. This is because the name being registered conflicts with an existing name. Please check if this is not expected.
  

DiffusionWrapper has 63.72 M params.
Keeping EMAs of 110.
Training UnifiedSegRDM as an unconditional model.
UnifiedSegRDM: max_segments=250, seg_npz_dir=None
  Diversity loss weight: 0.05
  Alignment loss weight: 0.05
  Normalize SAM tokens: False
  Embedding source: sam
  Alignment projection: 256 -> 256 (trainable=False)

Model parameters : 64,047,872
Running on       : cuda


In [10]:

# ── Optimizer + one forward pass ─────────────────────────────────────────────
import math

total_batch = args_dict["batch_size"] * args_dict["world_size"]
lr = args_dict["blr"] * total_batch
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=args_dict["weight_decay"])

# Move batch to device
batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

optimizer.zero_grad()
loss, loss_dict = model(x=None, c=None, batch=batch_gpu)
loss.backward()
optimizer.step()

print("Forward + backward pass OK")
print(f"  loss       : {loss.item():.4f}")
print(f"  loss_dict  : { {k: f'{v.item():.4f}' for k, v in loss_dict.items()} }")


Forward + backward pass OK
  loss       : 251.5987
  loss_dict  : {'train/loss_simple': '250.8733', 'train/loss_vlb': '10.2388', 'train/loss_diversity': '13.8155', 'train/loss_alignment': '0.6931', 'train/loss': '251.5987'}


In [11]:

# ── Debug training loop — 10 steps with real settings ─────────────────────────
import time

DEBUG_STEPS = 10
model.train()
optimizer.zero_grad()

print(f"Running {DEBUG_STEPS} debug steps  (batch_size={args_dict['batch_size']}, device={device})\n")
t0 = time.time()

for step, batch in enumerate(loader):
    if step >= DEBUG_STEPS:
        break

    batch_gpu = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    optimizer.zero_grad()
    loss, loss_dict = model(x=None, c=None, batch=batch_gpu)
    loss.backward()

    # gradient clip (same as engine_rdm.py)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    elapsed = time.time() - t0
    loss_str = "  ".join(f"{k}={v.item():.4f}" for k, v in loss_dict.items())
    print(f"  step {step+1:02d}/{DEBUG_STEPS}  loss={loss.item():.4f}  {loss_str}  [{elapsed:.1f}s]")

print(f"\nDone. {DEBUG_STEPS} steps in {time.time()-t0:.1f}s")


Running 10 debug steps  (batch_size=2, device=cuda)

  step 01/10  loss=313.9642  train/loss_simple=313.2506  train/loss_vlb=61.6607  train/loss_diversity=13.5856  train/loss_alignment=0.6856  train/loss=313.9642  [0.9s]
  step 02/10  loss=311.2601  train/loss_simple=310.5510  train/loss_vlb=13.9490  train/loss_diversity=13.5002  train/loss_alignment=0.6819  train/loss=311.2601  [1.4s]
  step 03/10  loss=200.2580  train/loss_simple=199.5482  train/loss_vlb=39.6096  train/loss_diversity=13.5056  train/loss_alignment=0.6913  train/loss=200.2580  [1.9s]
  step 04/10  loss=226.5113  train/loss_simple=225.7946  train/loss_vlb=75.8718  train/loss_diversity=13.6426  train/loss_alignment=0.6927  train/loss=226.5113  [2.5s]
  step 05/10  loss=267.7278  train/loss_simple=267.0136  train/loss_vlb=114.3460  train/loss_diversity=13.5951  train/loss_alignment=0.6900  train/loss=267.7278  [2.9s]
  step 06/10  loss=222.4568  train/loss_simple=221.7424  train/loss_vlb=38.8521  train/loss_diversity=13.5